# Lab 3 : Watch RAG fail on real-world edge cases

*W3 RAG Part 1 · Utrains LLMOps 8-Week Course*

Run each cell in order. Read the output. Move to the next.

See the matching slide in this week's concepts deck for the real-world story this lab teaches.


## What we are achieving in this lab

**Objective.** Lab 2's loop on a **messy** index. Same retrieve, same generate. Watch it fail in five named ways.

**Prerequisites.** Lab 2 finished. Same two keys: `OPENAI_API_KEY` and `ANTHROPIC_API_KEY`.

**The line so far.**

| Lab | What you can do |
|-----|-----------------|
| 1 | Chunk, embed, cosine, store, `retriever.invoke` |
| 2 | Retrieve, then Claude answers from those chunks |
| 3 (this lab) | See when that loop is not enough |

**What you will do.**

1. Index a small, hostile corpus (not the clean handbook).
2. Retrieve. **Print the hits.** Name the bug.
3. On a missing fact: generate with a vague prompt and with Lab 2's grounded prompt.

**What you should see.** Wrong neighbours in top-k. Two copies of the same policy. Two versions, no filter. A 2023 outage for a live question. A retriever that still returns chunks when the answer is not there.

**Cost.** A few embeddings plus two Claude calls. Fractions of a cent.

Week 4 is where you **fix** these. Today you only need to **recognise** them.


## Where this sits after Lab 2

Lab 2 worked because the handbook was clean: one topic per section, no duplicates, no old versions.

A real index is not. The loop does not change:

```
load → split → embed → store → retrieve → generate
```

The input does. Same `retriever.invoke`. Same Claude. Different documents → different failures.


### Step 1. Same keys as Lab 2


In [ ]:
import os
from dotenv import load_dotenv

load_dotenv()

if not os.getenv("OPENAI_API_KEY") or not os.getenv("ANTHROPIC_API_KEY"):
    raise EnvironmentError("Set OPENAI_API_KEY and ANTHROPIC_API_KEY in the repo-root .env file.")

print("keys : set")


### Step 2. Index a messy corpus (Lab 1)

Not `handbook.txt`. Seven short docs that will trip retrieve.


In [ ]:
from langchain_anthropic import ChatAnthropic
from langchain_core.documents import Document
from langchain_core.vectorstores import InMemoryVectorStore
from langchain_openai import OpenAIEmbeddings

EMBED_MODEL = "text-embedding-3-small"
embeddings = OpenAIEmbeddings(model=EMBED_MODEL)

CORPUS = [
    Document(
        page_content="Error E4221 occurs when the payment service receives a malformed VAT field.",
        metadata={"id": "e4221"},
    ),
    Document(
        page_content=(
            "Lunch this week is tomato soup, grilled cheese, and apple pie. "
            "Label your food. Recycling is by the stairs. "
            "Warehouse item PROD-A412-X3 is a replacement air filter. "
            "Do not leave dishes in the sink."
        ),
        metadata={"id": "buried-sku"},
    ),
    Document(
        page_content="Refund policy: 30 days. Submit a ticket within the window.",
        metadata={"id": "refund-v1-a", "version": "1"},
    ),
    Document(
        page_content="Refund policy: 30 days. Submit a ticket within the window.",
        metadata={"id": "refund-v1-b", "version": "1"},
    ),
    Document(
        page_content="Refund policy v2 (in effect from January 2026): 14 days.",
        metadata={"id": "refund-v2", "version": "2"},
    ),
    Document(
        page_content="Our annual outage in 2023 caused payment failures across the region.",
        metadata={"id": "incident-2023"},
    ),
    Document(
        page_content="PTO is unlimited with a 2-week annual minimum.",
        metadata={"id": "pto"},
    ),
]

vectorstore = InMemoryVectorStore.from_documents(CORPUS, embedding=embeddings)


def show_hits(query: str, k: int = 3) -> list[Document]:
    print("Question:", query)
    print()
    hits = vectorstore.as_retriever(search_kwargs={"k": k}).invoke(query)
    for i, hit in enumerate(hits, start=1):
        print(i, "[" + str(hit.metadata.get("id")) + "]", hit.page_content)
    print()
    return hits


print("stored", len(CORPUS), "docs")
print("embed :", EMBED_MODEL)


### 7.A  An exact token buried in the wrong topic

Lab 1: keyword search catches `PROD-A412-X3`. Cosine looks at the whole chunk. This chunk is mostly about lunch, so meaning search can miss the SKU.

Ask for the SKU. Read which rows come back.


In [ ]:
show_hits("What is SKU PROD-A412-X3?")


If `buried-sku` is not hit 1, cosine looked at "lunch" more than the token. Keywords would have matched `PROD-A412-X3` on that row.

Week 4 names the fix **hybrid** (BM25 + vectors). We do not build it today.


### 7.B  Duplicates steal top-k slots

Two copies of the 30-day refund sentence. `k=3`. They can take two slots. The 14-day policy may drop.


In [ ]:
show_hits("How long do I have to request a refund?", k=3)


If you see `refund-v1-a` and `refund-v1-b` both in the top three, you paid to store the same sentence twice.

Week 4: delete duplicates **before** you embed.


### 7.C  Two versions, no filter

30 days and 14 days are both "about refunds." Cosine cannot read `version` in metadata. Ask for all five hits.


In [ ]:
show_hits("How long do I have to request a refund?", k=5)


Both policies appear. The generator will pick one, blend them, or hedge. All three are wrong if only v2 is in force.

Week 4: filter on metadata (`version`, dates). Retrieval is cosine **plus** a filter, not cosine alone.


### 7.D  A neighbour that is about the words, not the job

The user means a live checkout failure. The 2023 outage is also "about payments failing." It will get retrieved.


In [ ]:
show_hits("Why did payments fail?")


If `incident-2023` is in the list, cosine did its job: same meaning. The **job** was "current incident," which this index does not know.

Week 4: dates and source type in metadata, or a reranker.


### 7.E  The answer is not in the index

There is no parental-leave doc here. Lab 2's handbook had one. Same question, different index.

The retriever still returns three chunks (`k=3`). PTO is a cousin of leave, so it often wins.

Then generate twice: a vague prompt (Week 2) and Lab 2's grounded prompt.


In [ ]:
hits = show_hits("What is the company's parental leave policy?")

parts = []
for hit in hits:
    parts.append(hit.page_content)
context = "\n\n---\n\n".join(parts)

llm = ChatAnthropic(model="claude-haiku-4-5", temperature=0)
question = "What is the company's parental leave policy?"

naive = llm.invoke(
    [
        ("system", "You are a helpful HR assistant. Answer the question."),
        ("user", "Context:\n" + context + "\n\nQuestion: " + question),
    ]
)
grounded = llm.invoke(
    [
        (
            "system",
            "You answer using ONLY the provided context. "
            "If the context does not contain the answer, say you cannot find it. Do not guess.",
        ),
        ("user", "Context:\n" + context + "\n\nQuestion: " + question),
    ]
)

print("--- vague prompt ---")
print(naive.content)
print()
print("--- grounded prompt (Lab 2) ---")
print(grounded.content)


The grounded prompt should refuse. A vague prompt may invent leave days from the PTO chunk. That is fluent and false.

A prompt is not a guarantee. Production later adds a cosine floor: if nothing is close enough, **your code** says "I don't know" and does not call the model. Lab 1: that floor is measured on your data.

## What you should be able to explain

> "The Lab 2 loop still fails when the index is messy: buried tokens, duplicates, two versions, stale neighbours, missing facts."

> "The retriever always returns k chunks. Wrong chunks, wrong answer — even with a good prompt."

> "I name the bugs. Hybrid, dedup, metadata filters, rerank, and a retrieve threshold are Week 4."

Week 3's question was: can I make a model speak about my data? **Yes — and this lab is the list of ways that sentence is incomplete.**
